# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the Olist **customers** dataset as part of the Data Quality Monitoring System for E-Commerce Operations. It performs data inspection, profiling, cleaning, validation, and export to ensure that customer records meet data quality standards before being loaded into PostgreSQL. The cleaned dataset serves as the customer dimension for subsequent SQL analysis and Power BI reporting.


### 1. Import the Libraries

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### 2. Load the Data

In [ ]:
customers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_customers_dataset.csv")
orders_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_orders_dataset.csv")
order_items_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_order_items_dataset.csv")
order_payments_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_order_payments_dataset.csv")
order_reviews_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_order_reviews_dataset.csv")
products_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_products_dataset.csv")
sellers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_sellers_dataset.csv")
geolocation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_geolocation_dataset.csv")
category_translation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\product_category_name_translation.csv")


## **CUSTOMERS DATASET**

### 1. Load Inspection

In [ ]:
# First 5 rows of the customers dataset
customers_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
# The number of rows and columns in the customers dataset
customers_df.shape

(99441, 5)

In [ ]:
# The names and data types of the columns in the customers dataset
customers_df.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

The dataset contains five columns. Four are correctly stored as text (customer_id, customer_unique_id, customer_city, and customer_state) because they represent identifiers or categorical values rather than numerical measurements.

The customer_zip_code_prefix column is currently stored as an integer (int64). Since postal codes function as identifiers and may contain leading zeros in some regions, they should be stored as a string (string) rather than a numeric data type. This prevents accidental mathematical operations and preserves the semantic meaning of the values.

In [ ]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [ ]:
# The summary statistics of the customers dataset
customers_df.describe()

,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


#### Observations

- The dataset contains 99,441 valid ZIP code prefix values, which is consistent with the total number of customer records. No missing values were identified.
- The ZIP code prefixes range from 1003 to 99990, indicating that customer records span a wide geographical area across Brazil.
- The reported mean (35,137.47), median (24,416), quartiles, and standard deviation (29,797.94) have limited business value because postal code prefixes are geographic identifiers, not quantitative measurements.
- Unlike variables such as price, freight cost, or delivery time, ZIP code prefixes should not be interpreted using measures of central tendency or dispersion. For example, an "average ZIP code" does not represent a meaningful business concept.

#### Recommendation

Convert customer_zip_code_prefix from int64 to string during the data preparation phase. Treating the field as a categorical geographic identifier rather than a numerical variable ensures that:

- PostgreSQL stores the column as a text-based identifier.
- Power BI classifies the field as a categorical attribute instead of a numeric measure.
- The column can be used reliably for grouping, filtering, and geographical analysis without encouraging inappropriate aggregations such as averages or sums.

In [ ]:
# The number of missing values in each column of the customers dataset
customers_df.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

#### Observations

The customers dataset contains no missing values across any of the five columns.
Every customer record includes a customer identifier, customer unique identifier, ZIP code prefix, city, and state.
The dataset achieves 100% completeness for all available attributes.

#### Data Quality Assessment

From a completeness perspective, the customers dataset is of high quality. Since no null values are present, no data imputation, row removal, or replacement strategies are required during the data preparation phase.

In [ ]:
# The number of duplicate values in each column of the customers dataset
for column in customers_df.columns:
    print(f"{column}: {customers_df[column].duplicated().sum()}")

customer_id: 0
customer_unique_id: 3345
customer_zip_code_prefix: 84447
customer_city: 95322
customer_state: 99414


The customer_id column contains no duplicate values, confirming that the primary key uniquely identifies every customer record.

The remaining columns contain duplicate values; however, these duplicates are expected and do not represent data quality problems. The customer_unique_id identifies repeat customers and therefore may legitimately appear in multiple records. Likewise, customer_zip_code_prefix, customer_city, and customer_state represent shared geographic attributes, making duplicate values both expected and appropriate.

Based on this assessment, no duplicate values require removal in the customers dataset.

### 2. Data Profiling

In [20]:
# The number of unique values in each column of the customers dataset
unique_values = pd.DataFrame({
    "Column": customers_df.columns,
    "Unique Values": customers_df.nunique().values
})

unique_values

,Column,Unique Values
0,customer_id,99441
1,customer_unique_id,96096
2,customer_zip_code_prefix,14994
3,customer_city,4119
4,customer_state,27


#### Data Quality Findings
- No null values.
- Primary key is unique.
- State codes appear complete.
- ZIP code prefix should be stored as text.
#### Business Insights
- 96,096 unique customers are represented.
- Customer records span all 27 Brazilian states.
- Customers are distributed across 4,119 cities and 14,994 ZIP code prefixes.
- Some customers appear in multiple records, indicating repeat purchasing activity.

In [ ]:
# Customer distribution across Brazilian states.
customers_df["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64


- All 27 Brazilian state abbreviations are represented.
- No unexpected or invalid state codes were identified.
- The state distribution appears realistic and consistent with the expected geographic coverage of the Olist marketplace.


In [ ]:
# Check if any city is associated with more than one state
city_state_check = (
    customers_df
    .groupby("customer_city")["customer_state"]
    .nunique()
    .sort_values(ascending=False)
)

city_state_check[city_state_check > 1]

customer_city
sao francisco    4
sao domingos     4
santa maria      4
vera cruz        4
bonito           4
                ..
prata            2
paraiso          2
guara            2
nova veneza      2
jussara          2
Name: customer_state, Length: 163, dtype: int64

In [23]:
customers_df[
    customers_df["customer_city"] == "bonito"
][["customer_city", "customer_state"]].drop_duplicates()

,customer_city,customer_state
12476,bonito,BA
19851,bonito,PE
23949,bonito,MS
35162,bonito,PA


The analysis identified 163 city names that appear in more than one Brazilian state.

This finding does not necessarily indicate a data quality issue. In Brazil, it is common for municipalities in different states to share the same city name. The combination of customer_city and customer_state provides the appropriate geographic context for uniquely identifying customer locations.

In [ ]:
# Check the prefix length of the customer_zip_code_prefix column
customers_df["customer_zip_code_prefix"].astype(str).str.len().value_counts()

customer_zip_code_prefix
5    75446
4    23995
Name: count, dtype: int64

In [ ]:
# Inspect the rows with a 4-digit zip code prefix
customers_df[
    customers_df["customer_zip_code_prefix"].astype(str).str.len() == 4
].head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
13,eabebad39a88bb6f5b52376faec28612,295c05e81917928d76245e842748184d,5704,sao paulo,SP
17,c5c61596a3b6bd0cee5766992c48a9a1,b6e99561fe6f34a55b0b7da92f8ed775,7124,guarulhos,SP
18,9b8ce803689b3562defaad4613ef426f,7f3a72e8f988c6e735ba118d54f47458,5416,sao paulo,SP
22,2938121a40a20953c43caa8c98787fcb,482441ea6a06b1f72fe9784756c0ea75,5713,sao paulo,SP
24,cb721d7b4f271fd87011c4c83462c076,a5844ba4bfc8d0cc61d13027c7e63bcc,8225,sao paulo,SP
25,f681356046d9fde60e70c73a18d65ea2,5f102dd37243f152aec3607970aad100,9121,santo andre,SP


The ZIP code prefix length assessment identified 75,446 values with five digits and 23,995 values with four digits. A manual inspection of these records showed values such as 1151, 4534, and 7124, primarily associated with customers located in São Paulo (SP). This indicates that the original leading zeros were removed when the CEP prefixes were stored as integer values.

To preserve the original geographic identifier, the customer_zip_code_prefix column will be converted to a string and padded with leading zeros using Python's str.zfill(5) method. This ensures that all CEP prefixes follow the standard five-character format before being loaded into PostgreSQL and used in Power BI.and Power BI.

In [31]:
# Check for whitespace in text columns
text_columns = ["customer_id", "customer_unique_id", "customer_city", "customer_state"]

for col in text_columns:
    whitespace = (customers_df[col] != customers_df[col].str.strip()).sum()
    print(f"{col}: {whitespace}")

customer_id: 0
customer_unique_id: 0
customer_city: 0
customer_state: 0


In [ ]:
# Check for empty strings in the customer_city column
(customers_df["customer_city"] == "").sum()

np.int64(0)

In [34]:
(customers_df["customer_city"] == customers_df["customer_city"].str.lower()).all()

np.True_

In [33]:
customers_df["customer_city"].sample(20)

97355    santa barbara d'oeste
31561                 cascavel
39727                  goiania
95765                    belem
65404                 cascavel
27278                    bauru
15217                  jundiai
77320       sao caetano do sul
62702                queimadas
25080                rio verde
87002                paracambi
3439                  teresina
82742              sao vicente
22330                  goiania
14861                sao paulo
59029                sao paulo
31332                 blumenau
94484                sao paulo
16679          pindamonhangaba
80187        santa cruz do sul
Name: customer_city, dtype: object

#### Data Profiling Summary

The data profiling process indicates that the customers dataset is structurally complete and of high quality. The dataset contains 99,441 customer records and five attributes, with no missing values or duplicate primary keys identified. All columns contain valid data types except for customer_zip_code_prefix, which is currently stored as an integer despite representing a geographic identifier.

The dataset contains 96,096 unique customers, indicating that some customers have placed multiple orders. This is an expected characteristic of an e-commerce dataset rather than a data quality issue. Geographic profiling shows that customers are distributed across 14,994 ZIP code prefixes, 4,119 cities, and all 27 Brazilian federal units, confirming nationwide customer coverage.

Further profiling identified 163 city names that occur in more than one state. A review of these records indicates that the duplicated city names correspond to legitimate municipalities located in different Brazilian states and therefore do not require correction. Text quality assessments confirmed that customer city names are consistently formatted in lowercase, contain no leading or trailing whitespace, and exhibit a standardized naming convention.

An assessment of the ZIP code prefixes revealed that 23,995 records contain four-digit values due to the loss of leading zeros when stored as integers. This is a formatting issue rather than a data quality defect and will be addressed during the data cleaning phase by converting the column to a five-character string while preserving leading zeros.

Overall, the profiling results indicate that the customers dataset requires minimal cleaning, with the primary transformation involving the formatting of ZIP code prefixes. No evidence of significant completeness, consistency, or validity issues was identified.

### 3. Data Cleaning

In [35]:
# Convert the customer_zip_code_prefix column to a string and pad with leading zeros to ensure a length of 5 characters
customers_df["customer_zip_code_prefix"] = (
    customers_df["customer_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

In [36]:
# Check the prefix length of the customer_zip_code_prefix column after padding
customers_df["customer_zip_code_prefix"].astype(str).str.len().value_counts()

customer_zip_code_prefix
5    99441
Name: count, dtype: int64

In [37]:
# Convert the text columns to string data type
text_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]

customers_df[text_columns] = customers_df[text_columns].astype("string")

In [38]:
# Check the text columns data types after conversion
customers_df.dtypes[text_columns]

customer_id           string[python]
customer_unique_id    string[python]
customer_city         string[python]
customer_state        string[python]
dtype: object

#### Data Cleaning Summary

The customers dataset required minimal transformation during the data cleaning phase. The primary modification involved converting the customer_zip_code_prefix column from an integer to a standardized five-character string to preserve leading zeros and maintain its role as a geographic identifier. No missing values, duplicate records, invalid state codes, inconsistent text formatting, or whitespace issues were identified. Consequently, no additional cleaning operations were necessary, and the dataset is ready for validation before loading into PostgreSQL.

### 4. Export the dataset

In [41]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [42]:
customers_df.to_csv(
    CLEANED_DATA_PATH / "olist_customers_dataset_cleaned.csv",
    index=False
)